# 🗺️ Módulo 09 - Notebook 01: Analítica geoespacial con GeoPandas

## 🌍 Introducción al análisis espacial de Datos

**Libro:** Saliendo de lo Pandito  
**Módulo:** 09 - Analítica Geoespacial GeoPandas  
**Duración estimada:** 70 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Entender** qué es GeoPandas y para qué sirve  
✅ **Crear** geometrías básicas (puntos, líneas, polígonos)  
✅ **Trabajar** con coordenadas geográficas  
✅ **Visualizar** datos espaciales en mapas  
✅ **Aplicar** análisis geoespacial a problemas de negocio

---

## 📋 Pre-requisitos

* ✅ Módulos 03-08 completados
* ✅ Conocimiento de Pandas DataFrame
* ✅ Familiaridad con mapas y coordenadas

---

## 📚 Contenido

1. Qué es GeoPandas
2. Geometrías Básicas (Point, LineString, Polygon)
3. GeoDataFrame
4. Sistemas de Coordenadas (CRS)
5. Visualización de Mapas
6. Caso Integrador: Mapa de Sucursales

---

## 💡 Por qué importa

**Analítica geoespacial transforma decisiones:**

* 🏪 **Retail:** ¿Dónde abrir la próxima sucursal?
* 🚚 **Logística:** Optimizar rutas de entrega
* 🏢 **Inmobiliario:** Valoración por ubicación
* 📈 **Marketing:** Segmentación territorial

**La ubicación es el dato más valioso**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (con coordenadas)
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    
    # Extraer ubicaciones únicas de sucursales
    df_sucursales = df_ventas[['sucursal_id', 'sucursal_nombre', 'zona', 'lat', 'lon']].drop_duplicates().reset_index(drop=True)
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros ventas: {len(df_ventas):,}")
    print(f"   🏪 Sucursales: {len(df_sucursales)}")
    print(f"   📅 Período: {df_ventas['fecha'].min().strftime('%Y-%m-%d')} a {df_ventas['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n🗺️ Coordenadas disponibles:")
    print(f"   • lat (latitud): {df_sucursales['lat'].min():.4f} a {df_sucursales['lat'].max():.4f}")
    print(f"   • lon (longitud): {df_sucursales['lon'].min():.4f} a {df_sucursales['lon'].max():.4f}")
    
    print(f"\n🎯 Este notebook usará coordenadas REALES")
    print(f"   Listo para análisis geoespacial con GeoPandas")
    
    print(f"\n📊 Vista previa de sucursales:")
    print(df_sucursales[['sucursal_nombre', 'zona', 'lat', 'lon']].head())
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    df_sucursales = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 GeoPandas: Pandas + Geometrías

### 🌍 ¿Qué es GeoPandas?

**GeoPandas** extiende Pandas para trabajar con **datos geoespaciales**.

**Relación:**
```
Pandas DataFrame
    +
Geometrías espaciales (puntos, líneas, polígonos)
    =
GeoDataFrame
```

---

### 📍 Geometrías Básicas

#### 1️⃣ **Point (Punto)**

Una ubicación única (latitud, longitud).

```python
from shapely.geometry import Point

# Ubicación de una sucursal
sucursal = Point(-68.8458, -32.8895)  # (lon, lat)
```

**Uso:** Tiendas, clientes, eventos

---

#### 2️⃣ **LineString (Línea)**

Una secuencia de puntos conectados.

```python
from shapely.geometry import LineString

# Ruta de entrega
ruta = LineString([(-68.85, -32.89), (-68.83, -32.87), (-68.81, -32.85)])
```

**Uso:** Rutas, calles, redes

---

#### 3️⃣ **Polygon (Polígono)**

Un área cerrada.

```python
from shapely.geometry import Polygon

# Zona de influencia
zona = Polygon([(-68.9, -32.9), (-68.8, -32.9), (-68.8, -32.8), (-68.9, -32.8)])
```

**Uso:** Barrios, zonas comerciales, países

---

### 📊 GeoDataFrame

**GeoDataFrame** = DataFrame + columna "geometry"

```python
import geopandas as gpd

# Crear GeoDataFrame
gdf = gpd.GeoDataFrame(
    {'nombre': ['Sucursal A', 'Sucursal B'],
     'ventas': [100000, 150000]},
    geometry=[Point(-68.85, -32.89), Point(-68.83, -32.87)]
)
```

**Resultado:**
```
    nombre    ventas                      geometry
0  Sucursal A  100000  POINT (-68.85000 -32.89000)
1  Sucursal B  150000  POINT (-68.83000 -32.87000)
```

---

### 🌎 Sistemas de Coordenadas (CRS)

**CRS** (Coordinate Reference System) define cómo interpretar coordenadas.

**Los 2 más comunes:**

| CRS | Código EPSG | Uso |
|-----|-------------|-----|
| **WGS84** | EPSG:4326 | GPS, Google Maps (lat/lon) |
| **Web Mercator** | EPSG:3857 | Mapas web (metros) |

**Transformación:**
```python
# De WGS84 (grados) a Web Mercator (metros)
gdf_metros = gdf.to_crs(epsg=3857)
```

---

### 🗺️ Visualización

**Mapa básico:**
```python
gdf.plot()
```

**Con color por categoría:**
```python
gdf.plot(column='ventas', cmap='YlOrRd', legend=True)
```

---

### 💼 Casos de Uso Empresariales

1. **Retail:** Mapa de sucursales con ventas
2. **Logística:** Rutas de entrega optimizadas
3. **Inmobiliario:** Propiedades en zonas premium
4. **Marketing:** Segmentación territorial
5. **Finanzas:** Sucursales bancarias y cobertura

---

### 🎯 Ventaja de GeoPandas

✅ **Operaciones espaciales:** Distancias, áreas, intersecciones  
✅ **Joins espaciales:** Cruzar datos por ubicación  
✅ **Visualización:** Mapas integrados  
✅ **Compatible con Pandas:** Misma sintaxis

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🗺️ GEOPANDAS: ANÁLISIS GEOESPACIAL")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    import geopandas as gpd
    from shapely.geometry import Point, LineString, Polygon
    print(f"Versión de GeoPandas: {gpd.__version__}")
except ImportError:
    print("⚠️  GeoPandas no instalado. Ejecuta: %pip install geopandas")

print("\n🎯 En este notebook aprenderás:")
print("  • Geometrías: Point, LineString, Polygon")
print("  • GeoDataFrame: DataFrame + geometrías")
print("  • CRS: Sistemas de coordenadas (WGS84, Web Mercator)")
print("  • Visualización de mapas")

print("\n📖 Métodos clave:")
print("  - Point(lon, lat)  # Crear punto")
print("  - gpd.GeoDataFrame(df, geometry=points)")
print("  - gdf.to_crs(epsg=3857)  # Transformar CRS")
print("  - gdf.plot()  # Visualizar mapa")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# 🎯 OPCIONAL: Usar datos georeferenciados reales de Unity Catalog

# Descomentar para usar datos reales:
"""
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

print("💾 Cargando datos georeferenciados desde Unity Catalog...")

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar datos con coordenadas GPS
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    
    # Obtener sucursales únicas (con coordenadas)
    df_sucursales = df_ventas[[
        'sucursal_id', 'sucursal_nombre', 'lat', 'lon', 'zona'
    ]].drop_duplicates()
    
    # Convertir a GeoDataFrame
    geometry = [Point(xy) for xy in zip(df_sucursales['lon'], df_sucursales['lat'])]
    gdf_sucursales = gpd.GeoDataFrame(
        df_sucursales, 
        geometry=geometry, 
        crs='EPSG:4326'  # WGS84 (coordenadas GPS estándar)
    )
    
    print(f"✅ Datos georeferenciados cargados:")
    print(f"   • Sucursales: {len(gdf_sucursales)}")
    print(f"   • Sistema de Coordenadas (CRS): {gdf_sucursales.crs}")
    print(f"   • Tipo de geometría: {gdf_sucursales.geometry.type.unique()[0]}")
    
    print(f"\n📊 Muestra de GeoDataFrame:")
    display(gdf_sucursales.head())
    
    print(f"\n💡 Variables disponibles:")
    print("   • df_ventas: DataFrame completo con todas las ventas")
    print("   • gdf_sucursales: GeoDataFrame con ubicaciones de sucursales")
    
    print(f"\n🗺️ Operaciones geoespaciales posibles:")
    print("   1. Crear buffers (zonas de influencia) alrededor de sucursales")
    print("   2. Calcular distancias entre sucursales")
    print("   3. Hacer spatial joins con otros datasets geográficos")
    print("   4. Visualizar en mapas interactivos")
    print("   5. Analizar coberturas y competencia territorial")
    
    # Ejemplo: Calcular distancias entre sucursales
    print(f"\n📏 Ejemplo - Distancia entre SUC001 y SUC002:")
    suc1 = gdf_sucursales[gdf_sucursales['sucursal_id'] == 'SUC001'].geometry.iloc[0]
    suc2 = gdf_sucursales[gdf_sucursales['sucursal_id'] == 'SUC002'].geometry.iloc[0]
    
    # Proyectar a sistema métrico para Argentina (EPSG:22185 - POSGAR 94)
    gdf_metric = gdf_sucursales.to_crs('EPSG:22185')
    suc1_m = gdf_metric[gdf_metric['sucursal_id'] == 'SUC001'].geometry.iloc[0]
    suc2_m = gdf_metric[gdf_metric['sucursal_id'] == 'SUC002'].geometry.iloc[0]
    distancia_metros = suc1_m.distance(suc2_m)
    
    print(f"   Distancia: {distancia_metros:.2f} metros ({distancia_metros/1000:.2f} km)")
    
except Exception as e:
    print(f"⚠️  Tabla no encontrada: {e}")
    print("   Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
"""

print("ℹ️  Usando datos sintéticos de este notebook")
print("="*70)

In [0]:
import pandas as pd

# Estructura demostrativa de datos geoespaciales
sucursales = pd.DataFrame({
    'Sucursal_ID': [1, 2, 3],
    'Nombre': ['Centro Financial Center', 'Norte Hub Comercial', 'Sur Plaza Center'],
    'Latitud': [-34.6037, -34.5453, -34.6500],
    'Longitud': [-58.3816, -58.4800, -58.4200],
    'Ventas_Mensuales': [450000, 320000, 280000]
})

print("--- Registro Geoespacial de Sucursales Comercial ---")
print(sucursales)



## 🎓 Conclusiones del notebook 09_01

### ✅ Lo que aprendiste

1. **GeoPandas extiende Pandas:**
   - `GeoDataFrame` = DataFrame + columna `geometry`
   - Sintaxis idéntica a Pandas, con operaciones espaciales adicionales
   - `gpd.GeoDataFrame(df, geometry=points, crs='EPSG:4326')`

2. **Geometrías básicas (Shapely):**
   - `Point(lon, lat)` — una ubicación (sucursal, cliente)
   - `LineString([...])` — una ruta o conexión
   - `Polygon([...])` — un área cerrada (zona de influencia, barrio)

3. **Sistemas de Coordenadas (CRS):**
   - `EPSG:4326` (WGS84): grados lat/lon, estándar GPS
   - `EPSG:3857` (Web Mercator): metros, mapas web
   - `gdf.to_crs(epsg=3857)` para transformar entre sistemas

4. **Visualización de mapas:**
   - `gdf.plot()` — mapa básico
   - `gdf.plot(column='ventas', cmap='YlOrRd', legend=True)` — mapa de calor
   - Plotly + GeoPandas para mapas interactivos

5. **Operaciones espaciales:**
   - `gdf.distance(otro_punto)` — distancia entre geometrías
   - `gdf.buffer(radio)` — zona de influencia alrededor de un punto
   - `gpd.sjoin(gdf1, gdf2)` — join espacial por ubicación

---

### 🎯 Reglas de Oro

👉 **Regla #1: Point es (lon, lat), NO (lat, lon)**
```python
# MALO: orden incorrecto
sucursal = Point(-32.8895, -68.8458)  # lat, lon → ubicación errónea

# BUENO: longitud primero, latitud después
sucursal = Point(-68.8458, -32.8895)  # lon, lat → correcto
```

👉 **Regla #2: Siempre definir el CRS al crear el GeoDataFrame**
```python
# MALO: sin CRS, las operaciones espaciales fallan o son incorrectas
gdf = gpd.GeoDataFrame(df, geometry=points)

# BUENO: especificar CRS desde el inicio
gdf = gpd.GeoDataFrame(df, geometry=points, crs='EPSG:4326')
# Transformar a sistema métrico para calcular distancias reales
gdf_metric = gdf.to_crs('EPSG:22185')  # POSGAR 94 (Argentina)
```

👉 **Regla #3: Para distancias reales, proyectar a CRS en metros**
```python
# MALO: distancia en grados (sin significado físico)
gdf_wgs84 = gpd.GeoDataFrame(df, geometry=points, crs='EPSG:4326')
dist = gdf_wgs84.geometry.iloc[0].distance(gdf_wgs84.geometry.iloc[1])
# Resultado en grados → inútil

# BUENO: proyectar a metros antes de medir
gdf_metros = gdf_wgs84.to_crs('EPSG:22185')
dist = gdf_metros.geometry.iloc[0].distance(gdf_metros.geometry.iloc[1])
# Resultado en metros → correcto
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Crear punto de una sucursal | `Point(lon, lat)` |
| Crear zona de influencia | `gdf.buffer(radio_metros)` |
| Calcular distancia entre sucursales | `gdf.to_crs(metros).distance(...)` |
| Visualizar sucursales en mapa | `gdf.plot(column='ventas', legend=True)` |
| Cruzar datos por ubicación | `gpd.sjoin(gdf1, gdf2, how='left')` |
| Coordenadas GPS (lat/lon) | `crs='EPSG:4326'` |
| Distancias/áreas en metros | `gdf.to_crs('EPSG:22185')` (Argentina) |
| Mapas web (tiles) | `gdf.to_crs('EPSG:3857')` |
| Mapa de calor por categoría | `gdf.plot(column='col', cmap='YlOrRd')` |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🗺️ ¡Analítica geoespacial con GeoPandas dominada!</h3>
  <p><i>"La ubicación es el dato más valioso: GeoPandas lo convierte en decisiones."</i></p>
</div>